# ChatGPT Archive Compiler — ephemeral Colab bootstrap v2

This is the preferred follow-up workflow after the first real-export diagnostic run. It clones the selected private GitHub branch into ephemeral storage under `/content`, installs the exact resolved commit, validates parent-authoritative graph normalization with synthetic multipart data, and optionally regenerates the real Archive IR plus privacy-safe diagnostics.

Only the source export and generated outputs persist in Google Drive. The repository checkout disappears when the Colab runtime ends. Before running, add a Colab secret named `GITHUB_TOKEN` with read-only Contents access to `jcollins-bioinfo/chatgpt-archive-compiler` and enable Notebook access. The token is passed through a temporary `GIT_ASKPASS` helper and is never placed in a URL, command argument, Git configuration, or notebook output.

Privacy boundary: Colab runs on Google-hosted infrastructure. The synthetic validation is enabled by default. Real-export ingestion remains disabled until you provide an exact ZIP path and the required acknowledgment. No cell prints conversation titles, message content, node identifiers, source filenames, warning locations, or exception text derived from the export.


In [ ]:
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

In [ ]:
from pathlib import Path

REPO_BRANCH = "agent/rebuild-colab-workflow"  # @param {type:"string"}
REPO_FULL_NAME = "jcollins-bioinfo/chatgpt-archive-compiler"
PUBLIC_REPO_URL = f"https://github.com/{REPO_FULL_NAME}.git"
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/ChatGPT Data Export")
OUTPUT_ROOT = DRIVE_PROJECT_DIR / "outputs"

if not DRIVE_PROJECT_DIR.is_dir():
    raise RuntimeError("Expected Drive folder is missing: MyDrive/ChatGPT Data Export")
if not REPO_BRANCH.strip():
    raise ValueError("REPO_BRANCH must not be empty.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Drive project directory: {DRIVE_PROJECT_DIR}")
print(f"Repository branch: {REPO_BRANCH}")

In [ ]:
import os
import re
import subprocess
import sys
import tempfile
from collections.abc import Iterator, Mapping, Sequence
from contextlib import contextmanager


def run_command(
    command: Sequence[str],
    *,
    cwd: Path | None = None,
    env: Mapping[str, str] | None = None,
    capture_output: bool = False,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    """Run one shell-free subprocess with explicit arguments."""

    return subprocess.run(
        list(command),
        cwd=cwd,
        env=dict(env) if env is not None else None,
        check=check,
        text=True,
        capture_output=capture_output,
    )


def get_github_token() -> str:
    """Read the private-repository token without exposing provider error text."""

    try:
        from google.colab import userdata

        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        raise RuntimeError(
            "Colab secret GITHUB_TOKEN is missing or Notebook access is disabled."
        ) from None
    if not token:
        raise RuntimeError("Colab secret GITHUB_TOKEN is empty.")
    return token


ASKPASS_SOURCE = """#!/usr/bin/env python3
import os
import sys

prompt = sys.argv[1].lower() if len(sys.argv) > 1 else ""
print("x-access-token" if "username" in prompt else os.environ["CAC_GIT_TOKEN"])
"""


@contextmanager
def authenticated_git_environment() -> Iterator[dict[str, str]]:
    """Yield a subprocess environment backed by a temporary askpass helper."""

    token = get_github_token()
    with tempfile.TemporaryDirectory(prefix="cac_git_auth_", dir="/content") as directory:
        helper = Path(directory) / "askpass.py"
        helper.write_text(ASKPASS_SOURCE, encoding="utf-8")
        helper.chmod(0o700)
        environment = os.environ.copy()
        environment.pop("GITHUB_TOKEN", None)
        environment.update(
            {
                "CAC_GIT_TOKEN": token,
                "GIT_ASKPASS": str(helper),
                "GIT_TERMINAL_PROMPT": "0",
            }
        )
        try:
            yield environment
        finally:
            environment.pop("CAC_GIT_TOKEN", None)


def resolve_remote_commit() -> str:
    """Resolve the selected branch to exactly one 40-character Git commit SHA."""

    valid_branch = run_command(
        ["git", "check-ref-format", "--branch", REPO_BRANCH],
        capture_output=True,
        check=False,
    ).returncode
    if valid_branch != 0:
        raise ValueError("REPO_BRANCH is not a valid Git branch name.")

    expected_ref = f"refs/heads/{REPO_BRANCH}"
    with authenticated_git_environment() as environment:
        result = run_command(
            [
                "git",
                "-c",
                "credential.helper=",
                "ls-remote",
                "--exit-code",
                PUBLIC_REPO_URL,
                expected_ref,
            ],
            env=environment,
            capture_output=True,
        )
    matches = [
        fields[0]
        for line in result.stdout.splitlines()
        if len(fields := line.split()) == 2 and fields[1] == expected_ref
    ]
    if len(matches) != 1 or re.fullmatch(r"[0-9a-f]{40}", matches[0]) is None:
        raise RuntimeError("Selected branch did not resolve to exactly one Git commit.")
    return matches[0]

In [ ]:
import importlib

CHECKED_OUT_COMMIT = resolve_remote_commit()
REPO_DIR = Path(
    tempfile.mkdtemp(
        prefix=f"chatgpt-archive-compiler-{CHECKED_OUT_COMMIT[:12]}-",
        dir="/content",
    )
)
with authenticated_git_environment() as environment:
    run_command(
        [
            "git",
            "-c",
            "credential.helper=",
            "clone",
            "--branch",
            REPO_BRANCH,
            "--single-branch",
            "--no-tags",
            PUBLIC_REPO_URL,
            str(REPO_DIR),
        ],
        env=environment,
    )
run_command(["git", "checkout", "--detach", CHECKED_OUT_COMMIT], cwd=REPO_DIR)
actual_commit = run_command(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, capture_output=True
).stdout.strip()
if actual_commit != CHECKED_OUT_COMMIT:
    raise RuntimeError("Ephemeral checkout did not resolve to the selected commit.")

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--disable-pip-version-check",
        "-e",
        f"{REPO_DIR}[notebooks]",
    ]
)
repository_source = (REPO_DIR / "src").resolve()
sys.path.insert(0, str(repository_source))
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "chatgpt_archive_compiler" or module_name.startswith(
        "chatgpt_archive_compiler."
    ):
        del sys.modules[module_name]

archive_compiler = importlib.import_module("chatgpt_archive_compiler")
ingest_module = importlib.import_module("chatgpt_archive_compiler.ingest")
serialization_module = importlib.import_module("chatgpt_archive_compiler.serialization")
IngestLimits = ingest_module.IngestLimits
SchemaMode = ingest_module.SchemaMode
ingest_export_zip = ingest_module.ingest_export_zip
inspect_zip = ingest_module.inspect_zip
summarize_archive = ingest_module.summarize_archive
read_archive_ir = serialization_module.read_archive_ir
write_archive_ir = serialization_module.write_archive_ir

imported_from = Path(archive_compiler.__file__).resolve()
if repository_source not in imported_from.parents:
    raise RuntimeError("Package import did not resolve to the ephemeral checkout.")
dirty = run_command(
    ["git", "status", "--porcelain", "--untracked-files=all"],
    cwd=REPO_DIR,
    capture_output=True,
).stdout
if dirty:
    raise RuntimeError("Package installation unexpectedly changed the Git checkout.")

print(f"Ephemeral checkout: {REPO_DIR}")
print(f"Commit: {CHECKED_OUT_COMMIT}")
print(f"Package version: {archive_compiler.__version__}")
print(f"Imported from: {imported_from}")

## Synthetic parent-only multipart validation

The source fixture deliberately omits every redundant `children` field. The assertions verify that canonical child edges are reconstructed from parent pointers, the original absence remains distinguishable in provenance, alternate branches survive, the visible path remains exact, multipart files are processed numerically, and serialization round-trips without warnings.


In [ ]:
import copy
import json
import zipfile

synthetic_conversation = {
    "id": "synthetic-parent-only",
    "title": "Synthetic parent-only branch graph",
    "create_time": 1_735_689_600,
    "update_time": 1_735_689_700,
    "current_node": "assistant-current",
    "mapping": {
        "root": {"id": "root", "parent": None, "message": None},
        "user": {
            "id": "user",
            "parent": "root",
            "message": {
                "id": "message-user",
                "author": {"role": "user"},
                "create_time": 1_735_689_600,
                "content": {"content_type": "text", "parts": ["Question"]},
                "metadata": {},
            },
        },
        "assistant-current": {
            "id": "assistant-current",
            "parent": "user",
            "message": {
                "id": "message-assistant-current",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_700,
                "content": {"content_type": "text", "parts": ["Current"]},
                "metadata": {},
            },
        },
        "assistant-alternate": {
            "id": "assistant-alternate",
            "parent": "user",
            "message": {
                "id": "message-assistant-alternate",
                "author": {"role": "assistant"},
                "create_time": 1_735_689_650,
                "content": {"content_type": "text", "parts": ["Alternate"]},
                "metadata": {},
            },
        },
    },
}

with tempfile.TemporaryDirectory(prefix="cac_synthetic_", dir="/content") as directory:
    temporary_path = Path(directory)
    fixture_zip = temporary_path / "multipart-parent-only.zip"
    with zipfile.ZipFile(fixture_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive_zip:
        for index in reversed(range(3)):
            conversation = copy.deepcopy(synthetic_conversation)
            conversation["id"] = f"synthetic-parent-only-{index}"
            archive_zip.writestr(
                f"conversations-{index:03d}.json",
                json.dumps([conversation], allow_nan=False),
            )
    synthetic_archive = ingest_export_zip(
        fixture_zip,
        limits=IngestLimits(),
        schema_mode=SchemaMode.STRICT,
    )
    synthetic_output = temporary_path / "archive.ir.json"
    write_archive_ir(synthetic_archive, synthetic_output)
    assert read_archive_ir(synthetic_output) == synthetic_archive

synthetic_summary = summarize_archive(synthetic_archive)
assert synthetic_summary.conversation_count == 3
assert synthetic_summary.node_count == 12
assert synthetic_summary.message_count == 9
assert synthetic_summary.current_path_message_count == 6
assert synthetic_summary.warning_count == 0
for conversation in synthetic_archive.conversations:
    nodes = {node.node_id: node for node in conversation.nodes}
    assert nodes["root"].children_node_ids == ["user"]
    assert nodes["user"].children_node_ids == [
        "assistant-alternate",
        "assistant-current",
    ]
    assert all(node.source_children_node_ids is None for node in nodes.values())
    assert conversation.current_path_node_ids == ["root", "user", "assistant-current"]

print("Synthetic parent-only multipart validation passed.")
print(f"Conversations: {synthetic_summary.conversation_count}")
print(f"Graph nodes: {synthetic_summary.node_count}")
print(f"Warnings: {synthetic_summary.warning_count}")

## Optional real-export regeneration — disabled by default

Provide the exact existing ZIP path in Drive, enable the Boolean control, and enter the acknowledgment exactly. This run writes a commit-stamped Archive IR and a small `warning_diagnostics.json` file. Diagnostics contain only structural counts and schema labels; they omit titles, messages, node IDs, filenames, paths within the source ZIP, warning locations, and source-derived excerpts.


In [ ]:
from collections import Counter
from datetime import UTC, datetime

RUN_REAL_EXPORT = False  # @param {type:"boolean"}
REAL_EXPORT_ZIP = ""  # @param {type:"string"}
PRIVACY_ACKNOWLEDGEMENT = ""  # @param {type:"string"}
REQUIRED_ACKNOWLEDGEMENT = "I UNDERSTAND THIS RUNS IN GOOGLE COLAB"

if not RUN_REAL_EXPORT:
    print("Real-export ingestion remains disabled.")
else:
    if PRIVACY_ACKNOWLEDGEMENT != REQUIRED_ACKNOWLEDGEMENT:
        raise RuntimeError("Exact privacy acknowledgment is required.")
    real_export_path = Path(REAL_EXPORT_ZIP).expanduser()
    if not real_export_path.is_file() or real_export_path.suffix.casefold() != ".zip":
        raise RuntimeError("REAL_EXPORT_ZIP must be an exact existing .zip file path.")

    diagnostic_limits = IngestLimits(max_warnings=100_000)
    try:
        source_manifest = inspect_zip(
            real_export_path,
            limits=diagnostic_limits,
            compute_hashes=False,
            compute_archive_hash=False,
        )
        candidate_files = [
            source_file
            for source_file in source_manifest.files
            if source_file.detected_kind.value
            in {"conversations_json", "conversations_json_candidate"}
        ]
        print(f"Safe ZIP members: {len(source_manifest.files)}")
        print(f"Conversation payload candidates: {len(candidate_files)}")
        print(f"Candidate JSON bytes: {sum(item.size_bytes for item in candidate_files):,}")
        real_archive = ingest_export_zip(
            real_export_path,
            limits=diagnostic_limits,
            schema_mode=SchemaMode.TOLERANT,
        )
    except Exception as exception:
        error_name = type(exception).__name__
        raise RuntimeError(
            f"Real-export ingestion failed safely ({error_name}); source-derived exception "
            "text was suppressed."
        ) from None

    run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    real_output_directory = OUTPUT_ROOT / "real" / f"{run_id}-{CHECKED_OUT_COMMIT[:12]}"
    real_output_directory.mkdir(parents=True, exist_ok=False)
    real_output_path = write_archive_ir(real_archive, real_output_directory / "archive.ir.json")
    real_summary = summarize_archive(real_archive)
    warnings = real_archive.all_warnings
    warning_codes = Counter(warning.code for warning in warnings)
    warning_severities = Counter(warning.severity.value for warning in warnings)
    affected_conversations = {
        code: len(
            {
                warning.conversation_id
                for warning in warnings
                if warning.code == code and warning.conversation_id is not None
            }
        )
        for code in sorted(warning_codes)
    }
    unknown_content_types = Counter(
        str(warning.context.get("content_type"))
        for warning in warnings
        if warning.code == "unknown_content_type"
    )
    source_edge_totals = Counter()
    for warning in warnings:
        if warning.code == "source_child_edges_disagree":
            for key in (
                "nodes_with_differences",
                "missing_source_edges",
                "conflicting_source_edges",
                "dangling_source_edges",
                "duplicate_source_edges",
            ):
                value = warning.context.get(key, 0)
                if isinstance(value, int):
                    source_edge_totals[key] += value
    diagnostics = {
        "commit": CHECKED_OUT_COMMIT,
        "archive_counts": {
            "conversations": real_summary.conversation_count,
            "nodes": real_summary.node_count,
            "messages": real_summary.message_count,
            "current_path_messages": real_summary.current_path_message_count,
        },
        "warning_count": len(warnings),
        "warnings_by_code": dict(sorted(warning_codes.items())),
        "warnings_by_severity": dict(sorted(warning_severities.items())),
        "affected_conversations_by_code": affected_conversations,
        "unknown_content_types": dict(sorted(unknown_content_types.items())),
        "source_child_edge_totals": dict(sorted(source_edge_totals.items())),
        "suppressed_count": sum(
            int(warning.context.get("suppressed_count", 0))
            for warning in warnings
            if warning.code == "warnings_suppressed"
        ),
    }
    diagnostics_path = real_output_directory / "warning_diagnostics.json"
    diagnostics_path.write_text(json.dumps(diagnostics, indent=2, sort_keys=True), encoding="utf-8")

    print("Real-export regeneration completed.")
    print(f"Payload members: {real_archive.metadata['conversation_payload_count']}")
    print(f"Conversations: {real_summary.conversation_count}")
    print(f"Graph nodes: {real_summary.node_count}")
    print(f"Messages: {real_summary.message_count}")
    print(f"Current-path messages: {real_summary.current_path_message_count}")
    print(f"Warnings: {real_summary.warning_count}")
    print(f"Warnings by code: {dict(sorted(warning_codes.items()))}")
    print(f"Archive IR: {real_output_path}")
    print(f"Diagnostics: {diagnostics_path}")

## Runtime retention

The checkout under `/content` is intentionally temporary and requires no cleanup in Drive. Generated real-export artifacts remain under `MyDrive/ChatGPT Data Export/outputs/real/<timestamp>-<commit>/`.
